# Data Cleaning


In [1]:
import sys
from pathlib import Path

# Find the project root by walking up until src/olist is visible, so the
# notebook runs from any working directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "olist").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from olist import cleaning, config as cfg, io_utils as io

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)
cfg.ensure_dirs()

raw = io.load_raw()
print(f"loaded {len(raw)} raw tables, {sum(len(d) for d in raw.values()):,} rows total")

loaded 9 raw tables, 1,550,922 rows total


## 1. Load raw data


In [2]:
orders_raw = raw["olist_orders_dataset"]
normalised = cleaning.normalise_columns(orders_raw)
print("before:", list(orders_raw.columns)[:4])
print("after :", list(normalised.columns)[:4])

# The rename is idempotent - running it twice cannot corrupt the names.
assert list(cleaning.normalise_columns(normalised).columns) == list(normalised.columns)
print("\nidempotent: PASS")

typed = cleaning.apply_dtypes(normalised)
pd.DataFrame({"raw_dtype": orders_raw.dtypes.values, "clean_dtype": typed.dtypes.astype(str).values},
             index=typed.columns)

before: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp']
after : ['Order_Id', 'Customer_Id', 'Order_Status', 'Order_Purchase_Timestamp']

idempotent: PASS


,raw_dtype,clean_dtype
Order_Id,object,string
Customer_Id,object,string
Order_Status,object,category
Order_Purchase_Timestamp,object,datetime64[ns]
Order_Approved_At,object,datetime64[ns]
Order_Delivered_Carrier_Date,object,datetime64[ns]
Order_Delivered_Customer_Date,object,datetime64[ns]
Order_Estimated_Delivery_Date,object,datetime64[ns]


## 2. Build clean tables


In [3]:
clean, audits = cleaning.build_clean_tables(raw)

for name, df in clean.items():
    print(f"{name:<40} {str(df.shape):>14}")

olist_customers_dataset                      (99441, 5)
olist_geolocation_dataset                    (19010, 6)
olist_orders_dataset                        (99441, 12)
olist_order_items_dataset                  (112650, 12)
olist_order_payments_dataset                (103886, 8)
olist_order_reviews_dataset                 (98167, 10)
olist_products_dataset                      (32951, 23)
olist_sellers_dataset                         (3095, 4)
product_category_name_translation               (72, 2)


## 3. Null report


In [4]:
null_report = []
for name, df in raw.items():
    null_share = df.isna().mean()
    for col, share in null_share[null_share > 0].items():
        null_report.append({"table": name, "column": col, "null_pct": round(share * 100, 2)})
null_report = pd.DataFrame(null_report).sort_values("null_pct", ascending=False)
null_report

,table,column,null_pct
0,olist_order_reviews_dataset,review_comment_title,88.34
1,olist_order_reviews_dataset,review_comment_message,58.70
4,olist_orders_dataset,order_delivered_customer_date,2.98
5,olist_products_dataset,product_category_name,1.85
6,olist_products_dataset,product_name_lenght,1.85
7,olist_products_dataset,product_description_lenght,1.85
8,olist_products_dataset,product_photos_qty,1.85
3,olist_orders_dataset,order_delivered_carrier_date,1.79
2,olist_orders_dataset,order_approved_at,0.16
9,olist_products_dataset,product_weight_g,0.01


In [5]:
orders_clean = clean["olist_orders_dataset"]
lifecycle = orders_clean[["Is_Approved", "Is_Shipped", "Is_Delivered"]].mean().mul(100).round(2)
print("lifecycle completion (% of orders):")
print(lifecycle.to_string(), "\n")

products_clean = clean["olist_products_dataset"]
imputed = [c for c in products_clean.columns if c.endswith("_Missing")]
print("imputation indicators kept on products:")
print(products_clean[imputed].sum().to_frame("rows_imputed").to_string())
print("\nnulls remaining in imputed columns:",
      int(products_clean[[c.replace("_Missing", "") for c in imputed]].isna().sum().sum()))

lifecycle completion (% of orders):
Is_Approved     99.84
Is_Shipped      98.21
Is_Delivered    97.02 

imputation indicators kept on products:
                                    rows_imputed
Product_Name_Lenght_Missing                  610
Product_Description_Lenght_Missing           610
Product_Photos_Qty_Missing                   610
Product_Weight_G_Missing                       2
Product_Length_Cm_Missing                      2
Product_Height_Cm_Missing                      2
Product_Width_Cm_Missing                       2

nulls remaining in imputed columns: 0


## 4. Normalise columns


In [6]:
reviews_raw = cleaning.normalise_columns(raw["olist_order_reviews_dataset"])
reviews_clean = clean["olist_order_reviews_dataset"]

print("reviews")
print(f"  raw rows                 : {len(reviews_raw):,}")
print(f"  orders with >1 review    : {(reviews_raw.groupby('Order_Id').size() > 1).sum():,}")
print(f"  clean rows               : {len(reviews_clean):,}")
print(f"  one row per order        : {reviews_clean['Order_Id'].is_unique}")
print(f"  max Reviews_Per_Order    : {reviews_clean['Reviews_Per_Order'].max()}")

geo_raw = cleaning.normalise_columns(raw["olist_geolocation_dataset"])
geo_clean = clean["olist_geolocation_dataset"]
print("\ngeolocation")
print(f"  raw rows                 : {len(geo_raw):,}")
print(f"  clean rows (1 per zip)   : {len(geo_clean):,}")
print(f"  coordinates outside Brazil dropped: "
      f"{len(geo_raw) - int(geo_raw['Geolocation_Lat'].between(-34, 5.5).sum()):,}")
print(f"  median points per zip    : {geo_clean['Geolocation_Points'].median():.0f}")

reviews
  raw rows                 : 99,224
  orders with >1 review    : 547
  clean rows               : 98,167
  one row per order        : True
  max Reviews_Per_Order    : 2

geolocation
  raw rows                 : 1,000,163
  clean rows (1 per zip)   : 19,010
  coordinates outside Brazil dropped: 31
  median points per zip    : 29


## 5. Timestamp audit


In [7]:
audits["timestamp_audit"]

,check,violations,share_pct
0,approved_before_purchase,0,0.0000
1,carrier_before_approved,1359,1.3666
2,customer_before_carrier,23,0.0231
3,delivered_but_status_not_delivered,6,0.0060
4,status_delivered_but_no_date,8,0.0080


In [8]:
anomaly_rate = orders_clean["Timestamp_Anomaly"].mean()
print(f"orders with at least one timestamp anomaly: "
      f"{orders_clean['Timestamp_Anomaly'].sum():,} ({anomaly_rate:.2%})")
print("kept and flagged, so any analysis can exclude them deliberately:")
print("    orders_clean.query('not Timestamp_Anomaly')")

orders with at least one timestamp anomaly: 1,396 (1.40%)
kept and flagged, so any analysis can exclude them deliberately:
    orders_clean.query('not Timestamp_Anomaly')


## 6. Items cleanliness


In [9]:
items_clean = clean["olist_order_items_dataset"]
outlier_cols = [c for c in items_clean.columns if c.endswith("_Is_Outlier")]
print(items_clean[outlier_cols].mean().mul(100).round(2).to_frame("flagged_pct").to_string())
print()
print("what the price tail actually looks like:")
print(items_clean.loc[items_clean["Price_Is_Outlier"], "Price"].describe().round(2).to_string())
print()
print("share of GMV sitting in flagged rows:",
      f"{items_clean.loc[items_clean['Price_Is_Outlier'], 'Price'].sum() / items_clean['Price'].sum():.1%}")

                          flagged_pct
Price_Is_Outlier                 7.48
Freight_Value_Is_Outlier        10.77

what the price tail actually looks like:
count    8427.00
mean      574.29
std       431.20
min       277.45
25%       329.00
50%       409.99
75%       649.00
max      6735.00

share of GMV sitting in flagged rows: 35.6%


## 7. Referential integrity


In [10]:
audits["referential_integrity"]

,relationship,orphan_rows,child_rows,orphan_pct,status
0,order_items.Order_Id -> orders,0,112650,0.0,PASS
1,payments.Order_Id -> orders,0,103886,0.0,PASS
2,reviews.Order_Id -> orders,0,98167,0.0,PASS
3,orders.Customer_Id -> customers,0,99441,0.0,PASS
4,order_items.Product_Id -> products,0,112650,0.0,PASS
5,order_items.Seller_Id -> sellers,0,112650,0.0,PASS


## 8. Reconciliation


In [11]:
audits["reconciliation"]

,metric,raw,clean,diff,diff_pct,status
0,orders,99441.00,99441.00,0.0,0.0,PASS
1,unique orders,99441.00,99441.00,0.0,0.0,PASS
2,order items,112650.00,112650.00,0.0,0.0,PASS
3,item price total,13591643.70,13591643.70,0.0,0.0,PASS
4,freight total,2251909.54,2251909.54,0.0,0.0,PASS
5,payment total,16008872.12,16008872.12,0.0,0.0,PASS
6,products,32951.00,32951.00,0.0,0.0,PASS
7,sellers,3095.00,3095.00,0.0,0.0,PASS


In [12]:
gate = audits["reconciliation"]
money_rows = gate[gate["metric"].str.contains("total")]
assert (money_rows["diff_pct"].abs() < 0.01).all(), money_rows.to_string(index=False)
print("PASS - no money leaked during cleaning.")

integrity = audits["referential_integrity"]
assert (integrity["orphan_rows"] == 0).all(), integrity.to_string(index=False)
print("PASS - every foreign key resolves.")

PASS - no money leaked during cleaning.
PASS - every foreign key resolves.


## 9. Save clean tables


In [13]:
for name, df in clean.items():
    path = io.save_parquet(df, cfg.CLEAN_DIR, name)
    print(f"{path.name:<45} {df.shape}")
print(f"\nwritten to {cfg.CLEAN_DIR}")

olist_customers_dataset.parquet               (99441, 5)
olist_geolocation_dataset.parquet             (19010, 6)
olist_orders_dataset.parquet                  (99441, 12)
olist_order_items_dataset.parquet             (112650, 12)
olist_order_payments_dataset.parquet          (103886, 8)
olist_order_reviews_dataset.parquet           (98167, 10)
olist_products_dataset.parquet                (32951, 23)
olist_sellers_dataset.parquet                 (3095, 4)
product_category_name_translation.parquet     (72, 2)

written to C:\Users\programmer\Desktop\program\ali-ajvand-data-portfolio\olist-delivery-analytics\data\processed\cleaning


## 10. Summary
